# GNN Marker Prediction — North Sea Dataset

Adapted from the Colorado GNN notebook.  
Key changes from the Colorado version:
- Well identifier column: `well_name` (string, e.g. `'11_10_1'`) instead of `wellName` (float)
- Depth column: `DS_INDEX` (float, 0.5-ft steps) instead of `DEPTH` (integer, 1-ft steps)
- Input features: `GR_CLEAN`, `RHOB_CLEAN`, `DTC_CLEAN`, `DEEPRES_CLEAN`, `NEUT_CLEAN` (may contain NaNs → imputed)
- Marker tops: 6 formations (`DUNLIN`, `HORDALAND`, `ROGALAND`, `SHETLAND`, `VESTLAND`, `VIKING`) → n_classes = 7
- Data is **pre-split** into train / valid sets (no need to split manually)
- `start_depth` is derived per-well from the data (no fixed cut-off like 6000 ft)
- `df_log` also carries a `marker_label` text column that can be used for sanity checks


## 1. Imports & Installation

In [ ]:
# Uncomment to install on a fresh environment
# !pip install numpy torch torchdata networkx
# !pip install dgl -f https://data.dgl.ai/wheels/repo.html

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import networkx as nx
import glob
import time
import os
import dgl
import dgl.data
import dgl.dataloading
import dgl.nn.pytorch as dglnn
import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, confusion_matrix, classification_report
)

print("torch:", torch.__version__)
print("dgl  :", dgl.__version__)
print("CUDA :", torch.cuda.is_available())

## 2. Load Data

Update `base_dir` to point to where your parquet files live.

In [ ]:
base_dir = "/path/to/your/north_sea_data/"   # <-- CHANGE THIS

df_log_train  = pd.read_parquet(os.path.join(base_dir, "log_train.parquet"))
df_tops_train = pd.read_parquet(os.path.join(base_dir, "tops_train.parquet"))
df_log_valid  = pd.read_parquet(os.path.join(base_dir, "log_valid.parquet"))
df_tops_valid = pd.read_parquet(os.path.join(base_dir, "tops_valid.parquet"))

print("df_log_train :", df_log_train.shape)
print("df_tops_train:", df_tops_train.shape)
print("df_log_valid :", df_log_valid.shape)
print("df_tops_valid:", df_tops_valid.shape)

In [ ]:
# Marker / formation names in the tops tables
# Adjust this list if your tops table has different columns
MARKER_COLS = ['DUNLIN', 'HORDALAND', 'ROGALAND', 'SHETLAND', 'VESTLAND', 'VIKING']

# Keep only the known marker columns (drop any extras)
df_tops_train = df_tops_train[MARKER_COLS]
df_tops_valid = df_tops_valid[MARKER_COLS]

# Number of classes = number of markers + 1 (background / below last marker)
N_CLASSES = len(MARKER_COLS) + 1   # 7

print("Marker columns:", MARKER_COLS)
print("N_CLASSES     :", N_CLASSES)
df_tops_train.head()

In [ ]:
# Input feature columns (may contain NaNs — handled in cleaning function below)
INPUT_FEATURES = ['GR_CLEAN', 'RHOB_CLEAN', 'DTC_CLEAN', 'DEEPRES_CLEAN', 'NEUT_CLEAN']

# Compute global medians from the training set to use as fallback for all-NaN columns
GLOBAL_MEDIANS = df_log_train[INPUT_FEATURES].median()

from sklearn.preprocessing import StandardScaler
SCALER = StandardScaler()
SCALER.fit(df_log_train[INPUT_FEATURES].fillna(GLOBAL_MEDIANS))

df_log_train.head()

## 3. Helper Functions

In [ ]:
def well_log_cleaning(
    well_name: str,
    df_log: pd.DataFrame,
    input_features: list,
    depth_col: str = 'DS_INDEX',
    well_col: str = 'well_name'
) -> pd.DataFrame:
    """
    Returns a cleaned dataframe for a single well.

    Changes vs. Colorado version
    ----------------------------
    - Well identifier is a string (e.g. '11_10_1') — no float-to-int conversion.
    - Depth column is DS_INDEX (float, 0.5-ft sampling) — kept as-is, not cast to int.
    - Features may have NaNs → forward-fill then backward-fill, then fill remaining
      with column median so the tensor is always finite.
    - No hardcoded GR=-1 replacement (that was Colorado-specific).
    """
    df_well = df_log[df_log[well_col] == well_name].copy()
    if df_well.empty:
        return df_well

    df_well = df_well.sort_values(depth_col).reset_index(drop=True)

    # Keep only the columns we need
    cols_to_keep = input_features + [depth_col]
    df_well = df_well[cols_to_keep]

    # Impute NaNs: forward-fill → backward-fill → column median
    df_well[input_features] = (
        df_well[input_features]
        .ffill()
        .bfill()
        .fillna(df_well[input_features].median())
    )

    # Fallback for completely all-NaN columns
    if 'GLOBAL_MEDIANS' in globals():
        df_well[input_features] = df_well[input_features].fillna(GLOBAL_MEDIANS)

    # Standardize and clip outliers
    if 'SCALER' in globals():
        df_well[input_features] = SCALER.transform(df_well[input_features])
        df_well[input_features] = df_well[input_features].clip(-10, 10)
    )

    # Drop rows that still have NaN in depth (shouldn't happen, but safety net)
    df_well = df_well.dropna(subset=[depth_col])

    return df_well

In [ ]:
def step_function(start_depth: float, end_depth: float, tops_depth: list) -> list:
    """
    Generates integer step-function labels for every 0.5-ft sample between
    start_depth and end_depth.

    Changes vs. Colorado version
    ----------------------------
    - Works with float depths (0.5-ft step) instead of integer feet.
    - Uses np.arange for the depth grid instead of range().
    - Tops are matched with a small tolerance (0.26 ft) to handle float precision.

    Returns
    -------
    List of (depth, label) tuples where label ∈ {1, 2, …, n_tops+1}.
    Returns [(start_depth, 1)] if tops list is empty or ordering is violated.
    """
    if not tops_depth or start_depth >= end_depth:
        return [(start_depth, 1)]

    if not all(tops_depth[i] < tops_depth[i + 1] for i in range(len(tops_depth) - 1)):
        return [(start_depth, 1)]

    STEP = 0.5
    TOL  = 0.26   # tolerance for floating-point depth matching

    depths = np.arange(start_depth, end_depth + STEP / 2, STEP)
    step_values = []
    current_label = 1
    tops_set = set(tops_depth)

    for d in depths:
        # Check whether any top falls within tolerance of this depth sample
        for t in tops_depth:
            if abs(d - t) < TOL:
                current_label += 1
                tops_depth = [x for x in tops_depth if x != t]  # consume matched top
                break
        step_values.append((round(d, 4), current_label))

    return step_values

In [ ]:
def create_graph(
    df_well: pd.DataFrame,
    step_values: list,
    depth_col: str = 'DS_INDEX',
    input_features: list = INPUT_FEATURES
) -> dgl.DGLGraph:
    """
    Creates a linear DGL graph from a cleaned well dataframe + step labels.

    Changes vs. Colorado version
    ----------------------------
    - Uses INPUT_FEATURES (5 logs) instead of [GR, Depth, Lat, Long].
    - `pos` is a (N, 1) tensor holding DS_INDEX (no lat/long available by default).
      If lat/long columns are present in df_well they are appended automatically.
    - Length mismatch between df_well and step_values is handled by truncating to
      the shorter of the two (previously this caused silent index errors).
    """
    if len(step_values) == 0:
        return dgl.graph(([], []))

    # Align lengths
    n = min(len(df_well), len(step_values))
    df_well    = df_well.iloc[:n].copy()
    step_values = step_values[:n]

    num_nodes = n
    if num_nodes == 0:
        return dgl.graph(([], []))

    # Linear chain edges
    src = list(range(num_nodes - 1))
    dst = list(range(1, num_nodes))
    g = dgl.graph((src, dst))

    # Features tensor  (N x len(input_features))
    feat_vals = df_well[input_features].values.astype(np.float32)
    g.ndata['features'] = torch.tensor(feat_vals, dtype=torch.float32)

    # Position tensor: depth + optional lat/long
    pos_cols = [depth_col]
    for c in ['Latitude', 'Longitude']:
        if c in df_well.columns:
            pos_cols.append(c)
    pos_vals = df_well[pos_cols].values.astype(np.float32)
    g.ndata['pos'] = torch.tensor(pos_vals, dtype=torch.float32)

    # Labels tensor
    labels = [t[1] for t in step_values]
    g.ndata['label'] = torch.tensor(labels, dtype=torch.int64)

    return g

## 4. Dataset & DataLoader

In [ ]:
class WellGraphDataset(Dataset):
    """
    PyTorch Dataset that wraps a set of wells into DGL graphs.

    Changes vs. Colorado version
    ----------------------------
    - Well name column is `well_name` (string) — no float/int casting.
    - Depth column is `DS_INDEX`.
    - `start_depth` defaults to None → per-well minimum depth is used.
    - Well list is built as the intersection of wells present in both
      df_log and df_tops (same logic, but string keys).
    - `max_wells` defaults to None (use all available wells).
    """

    def __init__(
        self,
        df_log: pd.DataFrame,
        df_tops: pd.DataFrame,
        input_features: list,
        depth_col: str = 'DS_INDEX',
        well_col: str = 'well_name',
        start_depth: float = None,
        max_wells: int = None
    ):
        self.df_log = df_log
        self.df_tops = df_tops
        self.input_features = input_features
        self.depth_col = depth_col
        self.well_col = well_col
        self.start_depth = start_depth

        # Build well list: intersection of df_log and df_tops
        log_wells  = set(df_log[well_col].unique())
        tops_wells = set(df_tops.index.tolist())
        well_list  = sorted(log_wells & tops_wells)

        if max_wells is not None:
            well_list = well_list[:max_wells]

        self.well_names = well_list
        print(f"WellGraphDataset: {len(self.well_names)} wells loaded.")

    def __len__(self):
        return len(self.well_names)

    def __getitem__(self, idx):
        well = self.well_names[idx]

        df_wgr = well_log_cleaning(
            well, self.df_log, self.input_features,
            depth_col=self.depth_col, well_col=self.well_col
        )

        if df_wgr.empty:
            return self._empty_graph()

        # Apply start_depth filter if provided
        if self.start_depth is not None:
            df_wgr = df_wgr[df_wgr[self.depth_col] > self.start_depth]

        if df_wgr.empty:
            return self._empty_graph()

        # Get tops for this well
        tops_list = (
            self.df_tops.loc[well].dropna().tolist()
            if well in self.df_tops.index
            else []
        )

        start_d = float(df_wgr[self.depth_col].min())
        end_d   = float(df_wgr[self.depth_col].max())

        labels = step_function(start_d, end_d, sorted(tops_list))

        graph = create_graph(
            df_wgr, labels,
            depth_col=self.depth_col,
            input_features=self.input_features
        )

        if graph.num_nodes() == 0:
            return self._empty_graph()

        return graph

    def _empty_graph(self):
        g = dgl.graph(([], []))
        g.add_nodes(1)
        g.ndata['features'] = torch.zeros(1, len(self.input_features), dtype=torch.float32)
        g.ndata['pos']      = torch.zeros(1, 1, dtype=torch.float32)
        g.ndata['label']    = torch.tensor([0], dtype=torch.int64)
        return g


def collate_graphs(samples):
    valid = [g for g in samples if g.num_nodes() > 0]
    if not valid:
        return dgl.graph(([], []))
    return dgl.batch(valid)

In [ ]:
# Build train and validation datasets
# max_wells=None uses all available wells; reduce for quick experiments

train_dataset = WellGraphDataset(
    df_log_train, df_tops_train,
    input_features=INPUT_FEATURES,
    max_wells=None
)

valid_dataset = WellGraphDataset(
    df_log_valid, df_tops_valid,
    input_features=INPUT_FEATURES,
    max_wells=None
)

train_dataloader = DataLoader(train_dataset, batch_size=4, shuffle=True,  collate_fn=collate_graphs)
valid_dataloader = DataLoader(valid_dataset, batch_size=4, shuffle=False, collate_fn=collate_graphs)

print(f"Train batches: {len(train_dataloader)} | Valid batches: {len(valid_dataloader)}")

## 5. GNN Model

Same `ImprovedGNN` architecture as the Colorado notebook — only `in_features` and `n_classes` change.

In [ ]:
class ImprovedGNN(torch.nn.Module):
    """
    Multi-layer GraphConv with BatchNorm + LeakyReLU + Dropout.
    Architecture unchanged from the Colorado version.
    """
    def __init__(self, in_feats, hidden_sizes, n_classes, dropout=0.1):
        super(ImprovedGNN, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.bns   = torch.nn.ModuleList()
        prev_size  = in_feats
        for h in hidden_sizes:
            self.convs.append(dglnn.GraphConv(prev_size, h))
            self.bns.append(torch.nn.BatchNorm1d(h))
            prev_size = h
        self.fc        = torch.nn.Linear(prev_size, n_classes)
        self.dropout   = torch.nn.Dropout(dropout)
        self.activation = F.leaky_relu

    def forward(self, g, features):
        h = features
        for conv, bn in zip(self.convs, self.bns):
            h = conv(g, h)
            h = bn(h)
            h = self.activation(h)
            h = self.dropout(h)
        return self.fc(h)

In [ ]:
# Optional: GAT variant
class ExtendedGAT(nn.Module):
    def __init__(self, in_feats, h_feats, num_classes, num_heads, dropout):
        super(ExtendedGAT, self).__init__()
        self.gat1 = dgl.nn.GATConv(in_feats, h_feats, num_heads)
        self.gat2 = dgl.nn.GATConv(h_feats * num_heads, h_feats, num_heads)
        self.gat3 = dgl.nn.GATConv(h_feats * num_heads, h_feats, num_heads)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.BatchNorm1d(h_feats * num_heads)
        self.norm2 = nn.BatchNorm1d(h_feats * num_heads)
        self.fc = nn.Linear(h_feats * num_heads, num_classes)

    def forward(self, g, in_feat):
        h = self.gat1(g, in_feat)
        h = F.elu(h)
        h = self.norm1(h.view(h.size(0), -1))
        h = self.dropout(h)
        h = self.gat2(g, h)
        h = F.elu(h)
        h = self.norm2(h.view(h.size(0), -1))
        h = self.dropout(h)
        h = self.gat3(g, h)
        h = F.elu(h)
        h = self.dropout(h)
        return self.fc(h.view(h.size(0), -1))

## 6. Loss, Optimizer & Scheduler

In [ ]:
in_features  = len(INPUT_FEATURES)     # 5
hidden_sizes = [128, 256, 256, 128, 64]
n_classes    = N_CLASSES               # 7

model = ImprovedGNN(in_features, hidden_sizes, n_classes, dropout=0.1)

# Class-balanced weights computed automatically
from collections import Counter
label_counts = Counter()
for g in train_dataset:
    if g.num_nodes() > 0:
        labs = g.ndata['label'].numpy()
        label_counts.update(labs.tolist())

# Adjusted labels are clamped to [0, n_classes-1] during training
adjusted_counts = Counter({k-1: v for k, v in label_counts.items()})
supports = torch.tensor([adjusted_counts.get(i, 0) for i in range(n_classes)], dtype=torch.float32)

computed_weights = torch.zeros_like(supports)
non_zero = supports > 0
if non_zero.any():
    computed_weights[non_zero] = supports[non_zero].max() / supports[non_zero]

criterion = torch.nn.CrossEntropyLoss(weight=computed_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

print(model)


In [ ]:
# Class-weight computation moved to previous cell.


## 7. Training

In [ ]:
def train_gnn(model, train_dataloader, optimizer, criterion, scheduler, n_classes, epochs=50):
    """
    Training loop — unchanged from the Colorado version.
    Labels are adjusted from 1-based to 0-based with clamp.
    """
    model.train()
    loss_history, accuracy_history = [], []
    precision_history, recall_history, f1_history = [], [], []

    for epoch in range(epochs):
        total_loss = 0.0
        all_preds, all_labels = [], []

        for batch_graphs in train_dataloader:
            if batch_graphs.num_nodes() == 0:
                continue
            batch_graphs = dgl.add_self_loop(batch_graphs)
            optimizer.zero_grad()

            features        = batch_graphs.ndata['features']
            labels          = batch_graphs.ndata['label']
            labels_adjusted = torch.clamp(labels - 1, 0, n_classes - 1)

            pred = model(batch_graphs, features)
            loss = criterion(pred, labels_adjusted)
            if torch.isnan(loss):
                print("NaN loss encountered. Skipping batch.")
                optimizer.zero_grad()
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()
            all_preds.extend(pred.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels_adjusted.cpu().numpy())

        avg_loss  = total_loss / max(len(train_dataloader), 1)
        accuracy  = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='weighted', zero_division=0)
        recall    = recall_score(all_labels, all_preds, average='weighted', zero_division=0)
        f1        = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

        loss_history.append(avg_loss)
        accuracy_history.append(accuracy)
        precision_history.append(precision)
        recall_history.append(recall)
        f1_history.append(f1)

        print(
            f"Epoch {epoch+1:3d}/{epochs} "
            f"→ Loss: {avg_loss:.4f}  Acc: {accuracy:.4f}  "
            f"P: {precision:.4f}  R: {recall:.4f}  F1: {f1:.4f}"
        )
        scheduler.step()

    return loss_history, accuracy_history, f1_history

In [ ]:
loss_hist, acc_hist, f1_hist = train_gnn(
    model, train_dataloader, optimizer, criterion, scheduler,
    n_classes=n_classes, epochs=120
)

## 8. Validation Evaluation

In [ ]:
def post_process_predictions(preds, window_size=5):
    """Median filter + Monotonicity enforcement."""
    processed = preds.copy()
    for i in range(len(preds)):
        start = max(0, i - window_size // 2)
        end   = min(len(preds), i + window_size // 2 + 1)
        processed[i] = np.median(preds[start:end])
        
    # Enforce monotonically non-decreasing labels
    for i in range(1, len(processed)):
        if processed[i] < processed[i-1]:
            processed[i] = processed[i-1]
            
    return processed.astype(int)


In [ ]:
model.eval()
all_raw_preds, all_proc_preds, all_true_labels = [], [], []

with torch.no_grad():
    for batch in valid_dataloader:
        if batch.num_nodes() == 0:
            continue
        batch = dgl.add_self_loop(batch)
        raw_preds = model(batch, batch.ndata['features']).argmax(dim=1).cpu().numpy()
        labels    = batch.ndata['label'].cpu().numpy()
        labels_adj = np.clip(labels - 1, 0, n_classes - 1)

        all_raw_preds.extend(raw_preds)
        all_proc_preds.extend(post_process_predictions(raw_preds, window_size=5))
        all_true_labels.extend(labels_adj)

all_raw_preds   = np.array(all_raw_preds)
all_proc_preds  = np.array(all_proc_preds)
all_true_labels = np.array(all_true_labels)

val_acc  = accuracy_score(all_true_labels, all_proc_preds)
val_prec = precision_score(all_true_labels, all_proc_preds, average='weighted', zero_division=0)
val_rec  = recall_score(all_true_labels, all_proc_preds, average='weighted', zero_division=0)
val_f1   = f1_score(all_true_labels, all_proc_preds, average='weighted', zero_division=0)

print(f"Validation — Acc: {val_acc:.4f}  P: {val_prec:.4f}  R: {val_rec:.4f}  F1: {val_f1:.4f}")
print()
print(classification_report(all_true_labels, all_proc_preds, zero_division=0))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_true_labels, all_proc_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(n_classes), yticklabels=range(n_classes))
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

## 9. Per-Well Visualisation

In [ ]:
def plot_well_side_by_side(
    df_well, actual_labels, predicted_labels,
    well_name=None, log_col='GR_CLEAN', depth_col='DS_INDEX'
):
    """
    Side-by-side scatter plot of actual vs predicted marker labels.

    Changes vs. Colorado version
    ----------------------------
    - x-axis is GR_CLEAN instead of GR.
    - y-axis is DS_INDEX instead of Depth.
    - colorbar tick range adapts to N_CLASSES.
    """
    log_vals = np.array(df_well[log_col])
    depth    = np.array(df_well[depth_col])
    actual   = np.array(actual_labels)
    predicted = np.array(predicted_labels)

    n_ticks   = n_classes
    boundaries = np.arange(0.5, n_ticks + 1.5, 1)
    # FIX: Use n_classes so colors don't max out at Cyan
    norm = colors.BoundaryNorm(boundaries, ncolors=n_classes)
    cmap = plt.get_cmap('tab10', n_classes)

    fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharey=True)

    sc1 = axes[0].scatter(log_vals, depth, c=actual, cmap=cmap, s=10, norm=norm)
    axes[0].invert_yaxis()
    axes[0].set_xlabel(log_col)
    axes[0].set_ylabel("Depth (DS_INDEX)")
    axes[0].set_title(f"Actual\n{well_name}" if well_name else "Actual")

    sc2 = axes[1].scatter(log_vals, depth, c=predicted, cmap=cmap, s=10, norm=norm)
    axes[1].invert_yaxis()
    axes[1].set_xlabel(log_col)
    axes[1].set_title("Predicted")

    fig.subplots_adjust(right=0.82)
    cbar_ax = fig.add_axes([0.85, 0.15, 0.03, 0.7])
    cbar = fig.colorbar(sc2, cax=cbar_ax, ticks=list(range(1, n_ticks + 1)))
    cbar.set_label("Marker Label")

    plt.suptitle(f"Well: {well_name}", fontsize=12)
    plt.tight_layout(rect=[0, 0, 0.82, 1])
    plt.show()

In [ ]:
def plot_well_from_name(
    well_name, df_log, df_tops, input_features, model,
    depth_col='DS_INDEX', well_col='well_name'
):
    """
    End-to-end: clean → step-function → graph → predict → plot.

    Changes vs. Colorado version
    ----------------------------
    - Uses well_log_cleaning() with DS_INDEX and string well_name.
    - start_depth is derived per-well (minimum DS_INDEX) not fixed at 6000.
    """
    df_wgr = well_log_cleaning(well_name, df_log, input_features, depth_col, well_col)
    df_wgr = df_wgr.dropna(subset=[depth_col])
    if df_wgr.empty:
        print(f"No valid depth data for well {well_name}")
        return

    tops_list = df_tops.loc[well_name].dropna().tolist() if well_name in df_tops.index else []
    start_d   = float(df_wgr[depth_col].min())
    end_d     = float(df_wgr[depth_col].max())
    step_vals = step_function(start_d, end_d, sorted(tops_list))

    graph = create_graph(df_wgr, step_vals, depth_col=depth_col, input_features=input_features)
    graph = dgl.add_self_loop(graph)

    actual_labels = graph.ndata['label'].cpu().numpy()

    model.eval()
    with torch.no_grad():
        preds = model(graph, graph.ndata['features'])
        predicted_labels = preds.argmax(dim=1).cpu().numpy()
        predicted_labels = post_process_predictions(predicted_labels, window_size=5)
        predicted_labels = predicted_labels + 1   # back to 1-based for display

    plot_well_side_by_side(df_wgr, actual_labels, predicted_labels, well_name=well_name,
                           depth_col=depth_col)

In [ ]:
# Example: visualise one validation well
example_well = valid_dataset.well_names[0]
plot_well_from_name(example_well, df_log_valid, df_tops_valid, INPUT_FEATURES, model)

## 10. Predicted Marker Depths & Recall Evaluation

In [ ]:
def findIndex(a, n, key):
    """Returns (first_occurrence_index, last_occurrence_index) of key in a."""
    start = -1
    for i in range(n):
        if a[i] == key:
            start = i
            break
    end = start
    if start >= 0:
        for i in range(n - 1, start - 1, -1):
            if a[i] == key:
                end = i
                break
    return (start, end)


def predicted_marker_depths(pred_array, depth_array, n_markers, con='last'):
    """
    Extracts one predicted depth per marker from the prediction array.

    Changes vs. Colorado version
    ----------------------------
    - Generalised to any number of markers (n_markers) instead of hardcoded 3.
    - Returns a dict {marker_index: depth} for easier DataFrame creation.
    - 'first' picks the first index where label == marker_class;
      'last' picks the last index.
    """
    arr = pred_array
    n   = len(arr)
    i   = 0 if con == 'first' else 1

    depths = {}
    for m in range(n_markers):
        label = m + 2   # marker classes start at 2 (1 = background before first marker)
        idx   = findIndex(arr, n, label)
        pos   = idx[i]
        depths[m] = float(depth_array[pos]) if (0 <= pos < len(depth_array)) else np.nan

    return depths

In [ ]:
# Generate predictions for all validation wells
n_markers   = len(MARKER_COLS)   # 6
valid_wells = valid_dataset.well_names
records     = []

for idx, well in enumerate(tqdm.tqdm(valid_wells, desc="Predicting")):
    df_wgr = well_log_cleaning(well, df_log_valid, INPUT_FEATURES)
    if df_wgr.empty:
        continue

    tops_list = df_tops_valid.loc[well].dropna().tolist() if well in df_tops_valid.index else []
    start_d   = float(df_wgr['DS_INDEX'].min())
    end_d     = float(df_wgr['DS_INDEX'].max())
    step_vals = step_function(start_d, end_d, sorted(tops_list))

    graph = create_graph(df_wgr, step_vals)
    if graph.num_nodes() == 0:
        continue
    graph = dgl.add_self_loop(graph)

    model.eval()
    with torch.no_grad():
        preds = model(graph, graph.ndata['features']).argmax(dim=1).cpu().numpy()
        preds = post_process_predictions(preds, window_size=5)

    depth_array = [t[0] for t in step_vals]
    pred_depths = predicted_marker_depths(preds, depth_array, n_markers, con='last')

    row = {'well_name': well}
    for m, col in enumerate(MARKER_COLS):
        row[col] = pred_depths[m]
    records.append(row)

df_preds = pd.DataFrame(records).set_index('well_name')
print(df_preds.shape)
df_preds.head()

In [ ]:
def recall_tops(df_tops_true, df_tops_pred, tolerance=10):
    """Unchanged from Colorado version."""
    if set(df_tops_true.columns) != set(df_tops_pred.columns):
        print("Column mismatch:", df_tops_true.columns.tolist(), "vs", df_tops_pred.columns.tolist())
        return None, None, None

    concat_df = df_tops_true.copy()
    for col in df_tops_pred.columns:
        concat_df[col + "_pred"] = df_tops_pred[col]

    tp, p, mae = 0, 0, 0
    for col in df_tops_true.columns:
        p   += concat_df[col].count()
        diff = (concat_df[col] - concat_df[col + "_pred"]).abs()
        tp  += (diff <= tolerance).sum()
        mae += diff.sum()

    return tp / p, mae / p, concat_df

In [ ]:
# Align actual tops to predicted wells
df_tops_actual = df_tops_valid.loc[
    df_tops_valid.index.isin(df_preds.index)
].copy()
df_tops_actual = df_tops_actual.reindex(df_preds.index)

recall, mae, df_results = recall_tops(df_tops_actual, df_preds, tolerance=20)
print(f"Recall (±20 ft): {recall:.4f}   MAE: {mae:.2f} ft")
df_results.head()

## 11. Learning Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(loss_hist, label='Loss')
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(acc_hist,  label='Accuracy')
axes[1].plot(f1_hist,   label='F1 (weighted)')
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Score")
axes[1].set_title("Training Metrics")
axes[1].legend()

plt.tight_layout()
plt.show()